# Watchtower model host
Runs Ollama on a Colab GPU. Models live on your Google Drive (`MyDrive/ollama`), so they download once.
Runtime > Change runtime type > T4 GPU. Then run all cells and copy the URL and token from the last cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/ollama', exist_ok=True)

In [ ]:
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
!curl -fsSL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared

In [ ]:
import subprocess, time
env = dict(os.environ, OLLAMA_MODELS='/content/drive/MyDrive/ollama', OLLAMA_HOST='127.0.0.1:11435')
subprocess.Popen(['ollama','serve'], env=env, stdout=open('/content/ollama.log','w'), stderr=subprocess.STDOUT)
time.sleep(5)
for m in ['qwen2.5-coder:7b','llama3.2:3b']:
    !OLLAMA_HOST=127.0.0.1:11435 OLLAMA_MODELS=/content/drive/MyDrive/ollama ollama pull {m}

In [ ]:
# Port 8080 is used by Colab itself, so we use 18080.
# Token-checking proxy: Ollama itself is never exposed, only this port, and only with the token.
import secrets, threading, httpx
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
TOKEN = secrets.token_urlsafe(32)
# Re-run safe: stop the server from a previous run of this cell, if any.
try:
    SRV.shutdown(); SRV.server_close()
except NameError:
    pass
class Srv(ThreadingHTTPServer):
    allow_reuse_address = True
    daemon_threads = True
class P(BaseHTTPRequestHandler):
    def _go(self):
        if self.headers.get('Authorization') != f'Bearer {TOKEN}':
            self.send_response(401); self.send_header('Content-Length','0'); self.end_headers(); return
        body = self.rfile.read(int(self.headers.get('Content-Length') or 0))
        r = httpx.request(self.command, 'http://127.0.0.1:11435' + self.path, content=body, timeout=900)
        self.send_response(r.status_code); self.send_header('Content-Length', str(len(r.content))); self.end_headers(); self.wfile.write(r.content)
    do_GET = do_POST = do_DELETE = _go
    def log_message(self, *a): pass
SRV = Srv(('127.0.0.1', 18080), P)
threading.Thread(target=SRV.serve_forever, daemon=True).start()

In [ ]:
import re
try:
    t.terminate()  # old tunnel from a previous run
except NameError:
    pass
t = subprocess.Popen(['cloudflared','tunnel','--url','http://127.0.0.1:18080'], stderr=subprocess.PIPE, text=True)
for line in t.stderr:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m: url = m.group(0); break
print('Paste this whole line in a Mac terminal and leave it open:\n\ncd "$HOME/Desktop/Watch tower" && COLAB_URL=%s COLAB_TOKEN=%s uv run python scripts/colab/bridge.py' % (url, TOKEN))
print('Keep this tab open. Colab stops idle sessions.')